## Step 00 (OPTIONAL) — Convert TikTok Watch History JSON to CSV

This notebook converts user data you can download directly from the TikTok app into a CSV list of all of the videos in an accounts watch history.


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

# If using google colab:
# from google.colab import drive
# drive.mount('/content/drive')

### Download your TikTok View History

Request your TikTok account data via the TikTok app [using this guide](https://www.tiktok.com/support/faq_detail?id=7543597460594285112&category=web_account). Make sure to:

- select `JSON` as your file format
- select the check box for `Your Activity` to include data on your video watch history

### Convert your TikTok View History Into a CSV

It can take up to a day or more for TikTok to respond to data requests. Once you have access to that JSON file, download it, drag and drop it into the `/raw` folder in this repository and rename it `history.json` before proceeding.


In [ ]:
# Locally:
INPUT_JSON_PATH = Path('../raw/history.json')
OUTPUT_CSV_PATH = Path('../raw/watch_history.csv')

# In Google Drive:
# INPUT_JSON_PATH = f"/content/drive/MyDrive/tiktok-content-reupload-tracker/raw/history.json"
# OUTPUT_CSV_PATH = f"/content/drive/MyDrive/tiktok-content-reupload-tracker/raw/watch_history.csv"

# Let's convert the column names to snake_case for easier handling in Python
def to_snake_case(name: str) -> str:
    cleaned = re.sub(r'[^0-9a-zA-Z]+', '_', str(name)).strip('_')
    return re.sub(r'(?<!^)(?=[A-Z])', '_', cleaned).lower()

with INPUT_JSON_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)

watch_history = data.get('Your Activity', {}).get('Watch History', {})
video_list = watch_history.get('VideoList') or watch_history.get('Video List') or []

if not isinstance(video_list, list):
    raise ValueError('Expected Watch History -> VideoList (or Video List) to be an array.')

df = pd.json_normalize(video_list)
df.columns = [to_snake_case(col) for col in df.columns]

OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f'Saved {len(df)} rows to {OUTPUT_CSV_PATH.resolve()}')
df.head()